# 08 — Silver Master Data

**Purpose:** Clean customer, merchant, and card master data from Bronze into Silver.

**Input:** `Capstone_team01.bronze.customer_master`, `merchant_master`, `card_master`  
**Output:** `Capstone_team01.silver.customer_master`, `merchant_master`, `card_master`  
**Quarantine:** `Capstone_team01.silver.quarantine_customer_master`, etc.

Run this notebook before transactions, authentication events, and CDC so reference checks can use the cleaned master tables.

Rules used:
- Required business key must be present.
- Duplicate business keys are quarantined in full; no arbitrary winner is selected.
- Date and numeric/boolean casts must succeed when a source value is present.
- Card customer references must exist in Silver customer master.
- No undocumented allowed-value lists or arbitrary numeric thresholds are imposed.


In [ ]:
from pyspark.sql import functions
from pyspark.sql.window import Window

spark.sql("USE CATALOG Capstone_team01")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")


In [ ]:
def process_master(source_table, target_table, key_column, date_columns=None,
                   decimal_columns=None, boolean_columns=None):
    date_columns = date_columns or []
    decimal_columns = decimal_columns or []
    boolean_columns = boolean_columns or []

    df = spark.table("Capstone_team01.bronze." + source_table)

    # Keep original values for cast validation and quarantine evidence.
    cast_columns = date_columns + decimal_columns + boolean_columns
    for column_name in cast_columns:
        df = df.withColumn("_raw_" + column_name, functions.col(column_name))

    for column_name in date_columns:
        df = df.withColumn(column_name, functions.to_date(functions.col(column_name)))

    for column_name in decimal_columns:
        df = df.withColumn(column_name, functions.col(column_name).cast("decimal(18,2)"))

    for column_name in boolean_columns:
        normalized = functions.lower(functions.trim(functions.col(column_name)))
        df = df.withColumn(
            column_name,
            functions.when(normalized.isin("true", "1"), functions.lit(True))
                     .when(normalized.isin("false", "0"), functions.lit(False))
                     .otherwise(functions.lit(None).cast("boolean"))
        )

    reasons = [
        functions.when(
            functions.col(key_column).isNull() |
            (functions.trim(functions.col(key_column)) == ""),
            functions.lit("MISSING_" + key_column.upper())
        )
    ]

    for column_name in date_columns + decimal_columns + boolean_columns:
        reasons.append(
            functions.when(
                functions.col("_raw_" + column_name).isNotNull() &
                functions.col(column_name).isNull(),
                functions.lit("INVALID_" + column_name.upper())
            )
        )

    duplicate_window = Window.partitionBy(key_column)
    df = df.withColumn("_key_count", functions.count("*").over(duplicate_window))

    reasons.append(
        functions.when(functions.col("_key_count") > 1,
                       functions.lit("DUPLICATE_" + key_column.upper()))
    )

    df = df.withColumn("_dq_reason", functions.concat_ws(";", *reasons))
    df = df.withColumn("_quarantined_at", functions.current_timestamp())

    quarantine_df = df.filter(functions.col("_dq_reason") != "")
    clean_df = df.filter(functions.col("_dq_reason") == "")

    # Card-to-customer reference validation.
    if source_table == "card_master":
        customers = spark.table("Capstone_team01.silver.customer_master").select("customer_id").distinct()
        missing_customers = (
            clean_df.join(customers, on="customer_id", how="left_anti")
                    .withColumn("_dq_reason", functions.lit("UNKNOWN_CUSTOMER_ID"))
                    .withColumn("_quarantined_at", functions.current_timestamp())
        )
        quarantine_df = quarantine_df.unionByName(missing_customers, allowMissingColumns=True)
        clean_df = clean_df.join(customers, on="customer_id", how="inner")

    for column_name in cast_columns:
        clean_df = clean_df.drop("_raw_" + column_name)

    clean_df = clean_df.drop("_key_count", "_dq_reason", "_quarantined_at")

    (
        clean_df.write.format("delta").mode("overwrite")
        .saveAsTable("Capstone_team01.silver." + target_table)
    )
    (
        quarantine_df.write.format("delta").mode("overwrite")
        .saveAsTable("Capstone_team01.silver.quarantine_" + target_table)
    )

    print("Created:", "Capstone_team01.silver." + target_table)
    print("Clean rows:", clean_df.count())
    print("Quarantined rows:", quarantine_df.count())


In [ ]:
# Customer master
process_master(
    source_table="customer_master",
    target_table="customer_master",
    key_column="customer_id",
    date_columns=["onboarded_date"]
)

# Merchant master
process_master(
    source_table="merchant_master",
    target_table="merchant_master",
    key_column="merchant_id",
    date_columns=["onboarded_date"]
)

# Card master
process_master(
    source_table="card_master",
    target_table="card_master",
    key_column="card_id",
    date_columns=["issue_date", "expiry_date"],
    decimal_columns=["daily_limit"],
    boolean_columns=["tokenized_flag"]
)


In [ ]:
# Verify the three Silver master tables
for table_name in ["customer_master", "merchant_master", "card_master"]:
    full_name = "Capstone_team01.silver." + table_name
    df = spark.table(full_name)
    print(full_name, "rows:", df.count())
    df.printSchema()
    display(df.limit(5))
